In [149]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.types.doc.document import PictureItem
from docling_core.transforms.serializer.markdown import MarkdownParams, ImageRefMode, MarkdownDocSerializer
from docling_core.transforms.serializer.markdown import MarkdownPictureSerializer
from docling_core.transforms.serializer.common import create_ser_result, SerializationResult

import inspect
import io
import base64
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List, Optional
import psycopg2
from pgvector.psycopg2 import register_vector
import json

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage

In [113]:
load_dotenv()

True

In [114]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [115]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [120]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [98]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.mode = "accurate"
pipeline_options.generate_table_images = True
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

In [99]:
result = converter.convert("pdfs/government-data-security-policies.pdf")

[INFO] 2026-04-08 14:46:26,465 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-08 14:46:26,481 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-08 14:46:26,481 [RapidOCR] main.py:53: Using C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-04-08 14:46:26,567 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-08 14:46:26,625 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-04-08 14:46:26,625 [RapidOCR] main.py:53: Using C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-04-08 14:46:26,663 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-08 14:46:26,715 [R

In [100]:
doc = result.document

In [101]:
class RefAwarePictureSerializer(MarkdownPictureSerializer):
    def serialize(self, item: PictureItem, **kwargs) -> SerializationResult:
        text_anchor = f"\n[IMAGE_REF: {item.self_ref}]\n"
        return create_ser_result(text=text_anchor, span_source=item)

class PictureAwareDocSerializer(ChunkingDocSerializer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.picture_serializer = RefAwarePictureSerializer()

class PictureAwareProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc) -> PictureAwareDocSerializer:
        return PictureAwareDocSerializer(doc=doc)

In [102]:
chunker = HybridChunker(
    max_tokens=512,
    merge_peers=True,
    split_on_limit=True,
    serializer_provider=PictureAwareProvider()
)

In [103]:
chunks = list(chunker.chunk(doc))

Token indices sequence length is longer than the specified maximum sequence length for this model (654 > 512). Running this sequence through the model will result in indexing errors


In [104]:
len(chunks)

41

In [141]:
class EnrichedChunk(BaseModel):
    chunk_id: int
    chunk_text: str
    chunk_images: Optional[List[str]] = None
    chunk_embeddings: List[float] 

In [136]:
def summarise_image(b64_image):
    system_msg = SystemMessage(
        content=(
            "You are a specialized RAG indexing assistant. Your task is to describe visual elements "
            "(tables, charts, or images) to make them highly searchable in a vector database.\n\n"
            "INSTRUCTIONS:\n"
            "1. IDENTIFY: Determine if the element is a table, chart, or image.\n"
            "2. IF A TABLE: I already have the raw markdown data. Your job is to provide the 'SO WHAT':"
            "What is the main conclusion or takeaway of this table?"
            "Identify the most important trend or outlier (e.g., 'Revenue peaked in Q3')."
            "Describe the context (e.g., 'This compares security protocols across regions')."
            "Do NOT transcribe the rows or columns.\n"
            "3. IF AN IMAGE/DIAGRAM: Describe the subject, setting, and any technical labels. "
            "Explain processes or relationships shown in diagrams.\n"
            "4. SEARCH OPTIMIZATION: Include industry-specific keywords and specific values "
            "that a user might search for.\n"
            "5. STYLE: Be dense and factual. Avoid 'This image shows' or 'This is a table'. "
            "Start directly with the description."
        )
    )
    human_msg = HumanMessage(
        content=[
            {"type": "text", "text": "Provide a dense, keyword-rich description of this image for RAG indexing."},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{b64_image}"}
            },
        ]
    )
    response = llm.invoke([system_msg, human_msg])
    return response.content
    

In [142]:
def process_chunk(chunk, doc, i):
    text = ""
    images = []

    headers = " > ".join(chunk.meta.headings)
    text += headers + "\n"

    elements = chunk.meta.doc_items 
    for el in elements:
        self_ref = el.self_ref
        parts = self_ref.lstrip("#/").split("/")
        el_type = parts[0]
        el_index = int(parts[1])

        if el_type == "texts":
            text += doc.texts[el_index].text + "\n"
        
        elif el_type == "tables":
            text += doc.tables[el_index].export_to_markdown(doc=doc) + "\n"
            pil_img = doc.tables[el_index].get_image(doc)
            if pil_img:
                buffered = io.BytesIO()
                pil_img.save(buffered, format="PNG")
                img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
                images.append(img_str)
                text += summarise_image(img_str)
        
        elif el_type == "pictures":
            pil_img = doc.pictures[el_index].get_image(doc)
            if pil_img:
                buffered = io.BytesIO()
                pil_img.save(buffered, format="PNG")
                img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
                images.append(img_str)
                text += summarise_image(img_str)
    
    return EnrichedChunk(
        chunk_id= i+1,
        chunk_text= text,
        chunk_images= images, 
        chunk_embeddings= embeddings.embed_query(text)
    )
        

In [ ]:
enriched_chunks = []
for i, chunk in enumerate(chunks):
    enriched = process_chunk(chunk, doc, i)
    enriched_chunks.append(enriched)

docker exec -it pgvector-db psql -U postgres

In [148]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [155]:
def sanitize_string(s: str) -> str:
    if s is None:
        return None
    return s.replace('\x00', '')

In [156]:
def store_chunk(chunk):
    chunk_embedding = chunk.chunk_embeddings
    chunk_text = sanitize_string(chunk.chunk_text)
    images = chunk.chunk_images
    highlights = None
    chunk_number = chunk.chunk_id
    try:
        cur.execute('''
            INSERT INTO chunk_store (chunk_embedding, chunk_text, images, highlights, chunk_number)
            VALUES (%s, %s, %s, %s, %s)
        ''', (chunk_embedding, chunk_text, images, json.dumps(highlights), chunk_number))
        conn.commit()
    except Exception as e: 
        print(f"Error: {e}")
        conn.rollback()

In [158]:
for chunk in enriched_chunks:
    store_chunk(chunk)